# 04 - Comprehensive Evaluation Framework, Classification Metrics & Research Roadmap

## Overview
This notebook synthesizes our evaluation methodology, provides mathematical definitions of performance metrics, documents historical evaluation anomalies, and outlines current research limitations and future directions.

We will cover:
1. Reproducing the complete held-out evaluation setup.
2. Mathematical breakdown of precision, recall, F1-score, support, macro average, and weighted average.
3. Analysis of the historical missing-class macro average anomaly.
4. Comparison of live interactive, held-out same-signer, and signer-independent evaluation paradigms.
5. Documenting `Current Limitations` of the current pipeline.
6. Outlining `Next Research Questions` for the project roadmap.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score, ConfusionMatrixDisplay

plt.style.use('ggplot')
%matplotlib inline

## 1. Experimental Setup & Held-Out Reproduction

We fit KNN ($k=5$) on `../training_data.npz` ($N=199$) and evaluate on `../test_data.npz` ($N=123$).

In [ ]:
train_data = np.load("../training_data.npz")
test_data = np.load("../test_data.npz")

X_train, y_train = train_data["X"], train_data["y"]
X_test, y_test = test_data["X"], test_data["y"]

knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train, y_train)

y_pred = knn.predict(X_test)
labels = sorted(np.unique(np.concatenate([y_train, y_test])))

print(f"Overall Accuracy: {accuracy_score(y_test, y_pred)*100:.2f}% ({int(np.sum(y_pred == y_test))}/{len(y_test)})")
print("
Classification Report:")
print(classification_report(y_test, y_pred, labels=labels, zero_division=0))

## 2. Mathematical Definition of Classification Metrics

To evaluate multiclass classifiers rigorously, we analyze per-class and aggregated metrics:

### 1. Precision ($P_c$)
Measures exactness — what fraction of predictions for class $c$ were correct?

$$P_c = rac{TP_c}{TP_c + FP_c}$$

### 2. Recall ($R_c$)
Measures completeness — what fraction of actual instances of class $c$ were correctly identified?

$$R_c = rac{TP_c}{TP_c + FN_c}$$

### 3. F1-Score ($F1_c$)
Harmonic mean of precision and recall:

$$F1_c = 2 \cdot rac{P_c \cdot R_c}{P_c + R_c} = rac{2 \cdot TP_c}{2 \cdot TP_c + FP_c + FN_c}$$

### 4. Support ($N_c$)
The number of true instances of class $c$ present in the test set.

### 5. Macro Average ($	ext{Macro } M$)
Unweighted mean of metric $M$ across all $C$ classes:

$$	ext{Macro } M = rac{1}{C} \sum_{c=1}^{C} M_c$$

*Note*: Treats all classes equally regardless of sample size $N_c$.

### 6. Weighted Average ($	ext{Weighted } M$)
Sample-size-weighted mean of metric $M$ across all classes:

$$	ext{Weighted } M = rac{\sum_{c=1}^{C} N_c \cdot M_c}{\sum_{c=1}^{C} N_c}$$

## 3. Case Study: The Missing-Class Macro Average Anomaly

In an earlier test collection run, class **L** was present in `y_train` but omitted from `y_test` ($N_{	ext{L}} = 0$).

All 113 captured test samples were predicted correctly ($113/113 = 100\%$ accuracy). However, the classification report yielded:
- Overall Accuracy: `1.00`
- Weighted Average F1: `1.00`
- **Macro Average F1: `0.92`**

### Mathematical Explanation:
1. Class `L` had $TP_{	ext{L}} = 0, FP_{	ext{L}} = 0, FN_{	ext{L}} = 0$. With `zero_division=0`, scikit-learn assigns $P_{	ext{L}} = 0.0, R_{	ext{L}} = 0.0, F1_{	ext{L}} = 0.0$.
2. Macro average computes the simple average across all 12 target classes:

$$	ext{Macro F1} = rac{11 	imes 1.00 + 1 	imes 0.00}{12} = rac{11.00}{12} pprox 0.9167 ightarrow \mathbf{0.92}$$

### Key Insight:
- Sample accuracy measures per-instance correctness ($113/113 = 100\%$).
- Macro average penalizes missing evaluation classes because it weights all classes equally regardless of support.
- Collecting 10 test samples for class `L` restored full support across all 12 classes, bringing Macro F1 to **1.00**.

## 4. Evaluation Paradigms Comparison

| Paradigm | Environment | Key Strengths | Limitations / Caveats |
|---|---|---|---|
| **Live Interactive Evaluation** | Real-time webcam video stream | Evaluates latency, temporal smoothing (debounce/latch), user responsiveness | Non-reproducible, subject to variable lighting and real-time signing speed |
| **Held-Out Same-Signer Evaluation** | Offline static snapshot datasets | Reproducible, isolated train/test sets, fast execution | Same signer anatomy and environment; does NOT demonstrate cross-signer generalization |
| **Signer-Independent Evaluation** | Cross-signer leave-one-signer-out splits | Gold standard for domain generalization and user readiness | Requires multi-signer data collection infrastructure |

## Current Limitations

1. **Same-Signer Train/Test Data**: Both training ($N=199$) and test ($N=123$) samples were recorded by the same individual under identical lighting and camera setup.
2. **Small Dataset Size**: Total dataset contains 322 landmark vectors across 12 classes.
3. **Limited Alphabet Subset**: Currently covers 12 static classes (`A`, `D`, `F`, `I`, `L`, `N`, `O`, `T`, `U`, `SPACE`, `BACKSPACE`, `CLEAR`).
4. **Omission of Dynamic Letters (J & Z)**: ASL letters **J** and **Z** require stroke motion trajectories and cannot be represented by a single static frame.
5. **Static Landmark Representation**: Single-frame 63D feature vectors ignore temporal context, velocity, and finger acceleration.
6. **Limited Rotation Robustness**: Normalization does not align hand orientation canonically, leaving feature vectors vulnerable to wrist roll or camera tilt.
7. **No Signer-Independent Split**: Lack of multi-signer data prevents validation of generalization to unseen hands.
8. **No Natural-Speed Continuous Benchmark**: Lacks sentence-level or continuous fingerspelling stream benchmarks.

## Next Research Questions

1. **How well does the model generalize to unseen signers?**
   - *Plan*: Collect test samples from multiple distinct signers to benchmark cross-signer accuracy drop.
2. **How robust is feature extraction to hand orientation and tilt?**
   - *Plan*: Test canonical orientation alignment (aligning wrist-to-middle-MCP along vertical axis) versus data augmentation.
3. **How should dynamic letters J and Z be modeled?**
   - *Plan*: Implement 2D/3D trajectory sequence tracking over multi-frame sliding windows.
4. **How do KNN and PyTorch MLP baselines compare?**
   - *Plan*: Train a PyTorch Multi-Layer Perceptron (MLP) on normalized landmarks and compare parameter footprint, inference latency, and soft probability outputs.
5. **What sequence model is appropriate for continuous fingerspelling?**
   - *Plan*: Evaluate Recurrent Neural Networks (LSTM/GRU), Temporal Convolutional Networks (TCN), or Transformers with CTC (Connectionist Temporal Classification) loss for unsegmented text decoding.
6. **What latency/accuracy tradeoffs matter for real-time interactive systems?**
   - *Plan*: Measure end-to-end frame-processing pipeline latency (MediaPipe extraction + model inference + debounce filtering) targeting $<30	ext{ ms}$ per frame.